# Track B — Fase 1–3: Training + OOF + Handoff

**BDC Satria Data 2026** | ConvNeXt-Tiny | 5-Fold CV

| | |
|---|---|
| Backbone | `convnext_tiny.in12k_ft_in1k` |
| Image size | 224 |
| Folds | 5 (Stratified Group K-Fold dari Track A) |
| GPU | T4 (Colab) |

> **Urutan:** Jalankan cell dari atas ke bawah. Jangan skip kecuali ada instruksi.

---
## 🔧 SETUP (Wajib Jalan Pertama)

In [ ]:
# Cell 0a — Clone / update repo
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull
print('✅ Repo siap')

In [ ]:
# Cell 0b — Install dependencies
!pip install -q timm scikit-learn
print('✅ timm + scikit-learn siap')

In [ ]:
# Cell 0c — Mount Drive + verifikasi artefak Track A
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
for p in [
    '/content/drive/MyDrive/BDC2026 apace/output_trackA/folds.csv',
    '/content/drive/MyDrive/BDC2026 apace/output_trackA/class_weights.npy',
]:
    status = '✅' if os.path.exists(p) else '❌ TIDAK ADA'
    print(status, p)

os.makedirs('/content/drive/MyDrive/BDC2026 apace/output_trackB', exist_ok=True)
print('✅ output_trackB siap')

In [ ]:
# Cell 0d — Pindah ke src dir + copy dataset.py Track A
import sys, os

SRC_DIR = '/content/satria-data-bdcugm02/track_b/src'
os.chdir(SRC_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Copy dataset.py dari Track A (read-only, jangan edit)
dataset_src = '/content/satria-data-bdcugm02/track_a/src/dataset.py'
assert os.path.exists(dataset_src), f'❌ Tidak ditemukan: {dataset_src}'
!cp "{dataset_src}" /content/satria-data-bdcugm02/track_b/src/dataset.py
print('✅ dataset.py tersedia')
print(f'✅ Working dir: {os.getcwd()}')

---
## 📦 FASE 1 — Baseline Single Fold

In [ ]:
# Fase 1 Task 1 — Verifikasi config merged
import sys; sys.modules.pop('config', None)
from config import CFG
import os

# Field lama
assert hasattr(CFG, 'accum_steps'), '❌ accum_steps hilang!'
assert hasattr(CFG, 'label_map'),   '❌ label_map hilang!'
assert CFG.accum_steps == 1
assert CFG.label_map[1] == 'Electronic'

# Field baru
assert hasattr(CFG, 'folds_csv'),          '❌ folds_csv belum ditambah!'
assert hasattr(CFG, 'class_weights_path'), '❌ class_weights_path belum ditambah!'
assert hasattr(CFG, 'save_dir'),           '❌ save_dir belum ditambah!'

assert CFG.seed == 42 and CFG.img_size == 224 and CFG.num_classes == 3

for p in [CFG.folds_csv, CFG.class_weights_path]:
    print(('✅' if os.path.exists(p) else '❌ TIDAK ADA:'), p)
os.makedirs(CFG.save_dir, exist_ok=True)
print('✅ config merged — field lama utuh + path baru ada')

In [ ]:
# Fase 1 Task 2 — Verifikasi scheduler
import sys; sys.modules.pop('scheduler', None)
import torch
from config import CFG
from scheduler import build_scheduler

m = torch.nn.Linear(4, 3)
opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr)
spe = 100
sch = build_scheduler(opt, steps_per_epoch=spe, cfg=CFG)

lrs = []
for _ in range(CFG.epochs * spe):
    lrs.append(opt.param_groups[0]['lr'])
    opt.step(); sch.step()

we = CFG.warmup_epochs * spe
assert lrs[0] < lrs[we - 1], '❌ warmup tidak naik'
assert lrs[-1] < lrs[we],    '❌ cosine tidak turun'
print(f'✅ LR: {lrs[0]:.2e} → puncak {max(lrs):.2e} → akhir {lrs[-1]:.2e}')

In [ ]:
# Fase 1 Task 3 Step 2 — Set env var FOLDS_CSV
import os
from config import CFG
os.environ['FOLDS_CSV'] = CFG.folds_csv
print('✅ FOLDS_CSV =', os.environ['FOLDS_CSV'])

In [ ]:
# Fase 1 Task 3 Step 3 — Load class_weights
import numpy as np
import torch
from config import CFG

cw = np.load(CFG.class_weights_path)
class_weights = torch.tensor(cw, dtype=torch.float32)
print('class_weights:', class_weights)
assert class_weights.shape == (3,), f'❌ shape: {class_weights.shape}'
assert class_weights.argmax().item() == 1, '❌ bobot tertinggi bukan kelas 1 (Electronic)'
print('✅ class_weights valid — kelas 1 (Electronic) tertinggi')

In [ ]:
# Fase 1 Task 3 Step 4 — Tarik 1 batch fold 0
import sys; sys.modules.pop('dataset', None)
from dataset import get_loaders
from config import CFG

train_loader, val_loader = get_loaders(fold=0, img_size=CFG.img_size, batch=CFG.batch)
images, labels = next(iter(train_loader))
print(f'images: {images.shape} | labels unik: {sorted(labels.unique().tolist())}')
assert images.shape[1:] == (3, 224, 224)
assert labels.min() >= 0 and labels.max() <= 2
print(f'train batches: {len(train_loader)} | val batches: {len(val_loader)}')
print('✅ Loader asli jalan')

In [ ]:
# Fase 1 Task 3 Step 5 — VISUAL CHECK ⏸️ (cek mata sebelum lanjut)
import matplotlib.pyplot as plt
import torch
from config import CFG

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = (images[i].cpu() * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(CFG.class_names[labels[i].item()])
    ax.axis('off')
plt.tight_layout(); plt.show()
print('👁️  CEK MATA: gambar sampah nyata? Label cocok isi?')
print('⏸️  Kalau OK → lanjut ke cell berikutnya')

In [ ]:
# Fase 1 Task 4 Step 2 — Smoke test 1 epoch
import sys
for m in ['train', 'dataset', 'scheduler', 'model', 'losses_metrics', 'config', 'seed_utils']:
    sys.modules.pop(m, None)

import numpy as np, torch
from config import CFG

cw = np.load(CFG.class_weights_path)
class_weights = torch.tensor(cw, dtype=torch.float32)

from train import run_training
f1, mins = run_training(fold=0, cfg=CFG, class_weights=class_weights, max_epochs=1)
print(f'\n✅ Smoke test lolos | F1 1-epoch: {f1:.4f} | {mins:.1f} mnt/epoch')
print('⏸️  Kalau OK → lanjut training penuh')

In [ ]:
# Fase 1 Task 5 Step 1 — Training penuh fold 0
import sys; sys.modules.pop('train', None)
import numpy as np, torch
from config import CFG
from train import run_training

cw = np.load(CFG.class_weights_path)
class_weights = torch.tensor(cw, dtype=torch.float32)

best_f1, mins_per_epoch = run_training(fold=0, cfg=CFG, class_weights=class_weights)
est_5fold = mins_per_epoch * CFG.epochs * 5 / 60
print(f'\n📊 BASELINE fold 0 Macro-F1: {best_f1:.4f}')
print(f'⏱️  {mins_per_epoch:.1f} mnt/epoch → estimasi 5-fold: {est_5fold:.1f} jam GPU')
if est_5fold > 12:
    print('⚠️  >12 jam — bagi fold antar mesin atau kurangi epochs')
elif est_5fold > 6:
    print('⚠️  Hati-hati disconnect Colab — checkpoint aman di Drive')
else:
    print('✅ Feasible sekali jalan')

In [ ]:
# Fase 1 Task 5 Step 2 — Simpan baseline log
import json
from config import CFG

log = {
    'phase': 'fase1_baseline',
    'fold': 0,
    'baseline_macro_f1': float(best_f1),
    'minutes_per_epoch': mins_per_epoch,
    'accum_steps': CFG.accum_steps,
    'backbone': CFG.backbone,
    'config': {k: (v if isinstance(v, (int, float, str, list)) else str(v)) for k, v in vars(CFG).items()},
}
# Simpan juga sebagai fold0_log.json agar resume-friendly di Fase 2
for fname in ['fold0_baseline_log.json', 'fold0_log.json']:
    path = f'{CFG.save_dir}/{fname}'
    payload = log if fname == 'fold0_baseline_log.json' else {
        'fold': 0,
        'best_macro_f1': float(best_f1),
        'minutes_per_epoch': mins_per_epoch,
        'accum_steps': CFG.accum_steps,
    }
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2)
print(f'✅ fold0_baseline_log.json + fold0_log.json tersimpan di {CFG.save_dir}')

---
## 🔁 FASE 2 — Full 5-Fold Training + OOF

In [ ]:
# Fase 2 Task 1 — Estimasi waktu total
import json
from config import CFG

with open(f'{CFG.save_dir}/fold0_baseline_log.json') as f:
    log = json.load(f)
mins = log['minutes_per_epoch']
total = mins * CFG.epochs * 5 / 60
print(f'{mins:.1f} mnt/epoch × {CFG.epochs} epoch × 5 fold = {total:.1f} jam total')
if total > 12:
    print('→ WAJIB bagi fold antar mesin/akun')
elif total > 6:
    print('→ Bagi 2 mesin lebih aman (Colab bisa disconnect)')
else:
    print('→ Bisa sequential 1 session')
print('\n⏸️  PUTUSKAN: fold mana dijalankan di mesin ini, lalu edit list di cell berikutnya')

In [ ]:
# Fase 2 Task 2 — Jalankan fold (edit list sesuai mesin ini)
import os, sys
from config import CFG
os.environ['FOLDS_CSV'] = CFG.folds_csv

for m in ['run_all_folds', 'train', 'config', 'dataset']:
    sys.modules.pop(m, None)
from run_all_folds import run_folds

# ⚠️  EDIT LIST INI sesuai pembagian mesin
# fold 0 sudah selesai dari Fase 1 → otomatis di-skip
FOLDS_MESIN_INI = [0, 1, 2, 3, 4]

results = run_folds(FOLDS_MESIN_INI)
print('\n📊 Hasil:', results)

In [ ]:
# Fase 2 Task 2 Step 4 — Verifikasi 5 checkpoint lengkap
import os
from config import CFG

missing = [f for f in range(5) if not os.path.exists(f'{CFG.save_dir}/fold{f}.pt')]
assert not missing, f'❌ Kurang fold: {missing}'
print('✅ 5 checkpoint lengkap')

In [ ]:
# Fase 2 Task 3 — Kumpulkan prediksi OOF
import os, sys
from config import CFG
os.environ['FOLDS_CSV'] = CFG.folds_csv

for m in ['collect_oof', 'dataset', 'model']:
    sys.modules.pop(m, None)
from collect_oof import collect_oof

oof_f1 = collect_oof()

In [ ]:
# Fase 2 Task 3 Step 3 — Sanity gap CV vs OOF
import json, statistics
from config import CFG

fold_f1s = []
for f in range(5):
    with open(f'{CFG.save_dir}/fold{f}_log.json') as fp:
        fold_f1s.append(json.load(fp)['best_macro_f1'])

mean_f1 = statistics.mean(fold_f1s)
std_f1  = statistics.stdev(fold_f1s)
print(f'CV per fold: {[f"{x:.4f}" for x in fold_f1s]}')
print(f'CV Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}')
print(f'OOF overall: {oof_f1:.4f}')
gap = abs(mean_f1 - oof_f1)
assert gap < 0.05, f'❌ Gap {gap:.4f} terlalu besar — kemungkinan index OOF salah!'
print('✅ CV dan OOF konsisten')

In [ ]:
# Fase 2 Task 4 — Simpan cv_summary.json
import json
from config import CFG

summary = {
    'backbone': CFG.backbone,
    'folds': {f'fold{i}': fold_f1s[i] for i in range(5)},
    'cv_mean_macro_f1': mean_f1,
    'cv_std_macro_f1': std_f1,
    'oof_overall_macro_f1_argmax': float(oof_f1),
    'epochs': CFG.epochs,
    'lr': CFG.lr,
    'img_size': CFG.img_size,
    'batch': CFG.batch,
    'accum_steps': CFG.accum_steps,
    'seed': CFG.seed,
    'loss': f'weighted CE + label_smoothing {CFG.label_smoothing}',
}
with open(f'{CFG.save_dir}/cv_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print('✅ cv_summary.json tersimpan')

---
## 🤝 FASE 3 — Handoff ke Track C + Report

In [ ]:
# Fase 3 Task 1 — Verifikasi handoff dari sudut pandang Track C
import sys
for m in ['verify_handoff', 'config']:
    sys.modules.pop(m, None)
from verify_handoff import verify
assert verify(), 'Perbaiki artefak yang ❌ dulu'

In [ ]:
# Fase 3 Task 2 Step 1 — Rekam versi library
import json, sys
import torch, timm, sklearn, numpy, pandas
from config import CFG

env = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'timm': timm.__version__,
    'sklearn': sklearn.__version__,
    'numpy': numpy.__version__,
    'pandas': pandas.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
with open(f'{CFG.save_dir}/environment.json', 'w') as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))

In [ ]:
# Fase 3 Task 2 Step 2 — Tulis REPRODUCE.md
from config import CFG

content = f"""# Reproduksi Hasil Track B — BDC Satria Data 2026

## Ringkas
Fine-tune `{CFG.backbone}` (pre-trained, via timm) untuk klasifikasi 3 kelas sampah.
Stratified-Group 5-Fold CV (folds.csv dari Track A, seed 42, group-aware anti-leakage).

## Urutan reproduksi
1. Clone repo: `git clone https://github.com/agaggigit/satria-data-bdcugm02.git`
2. Mount Drive, set `FOLDS_CSV` env var ke path folds.csv di Drive
3. `pip install timm scikit-learn`
4. `cp track_a/src/dataset.py track_b/src/dataset.py`
5. Jalankan notebook `track_b/notebooks/02_fase1_3_training.ipynb`
   - Atau: `run_all_folds.run_folds([0,1,2,3,4])` → 5 checkpoint
   - Lalu: `collect_oof.collect_oof()` → oof.npy + metrik

## Konfigurasi kunci
- Seed 42 (torch/numpy/random + cudnn deterministic) — di-set ulang tiap fold
- Image {CFG.img_size}, batch {CFG.batch}, epochs {CFG.epochs}
- AdamW lr {CFG.lr} wd {CFG.weight_decay}, warmup {CFG.warmup_epochs} epoch + cosine → {CFG.min_lr}
- Weighted CrossEntropy (class_weights.npy Track A) + label smoothing {CFG.label_smoothing}
- AMP fp16 (torch.amp) + grad clipping max_norm {CFG.max_grad_norm}
- Grad checkpointing ON (VRAM)

## Bukti proses (bukan manual)
- Log per fold: fold{{N}}_log.json (metrik per fold)
- Log environment: environment.json
- Kurva loss/F1 per epoch tercetak di notebook (tersimpan di history Colab)
- Kode lengkap di Git repo tim
"""
with open(f'{CFG.save_dir}/REPRODUCE.md', 'w') as f:
    f.write(content)
print('✅ REPRODUCE.md tersimpan')

In [ ]:
# Fase 3 Task 3 — Generate draft report dari angka asli
import json
from config import CFG

with open(f'{CFG.save_dir}/cv_summary.json') as f:
    s = json.load(f)

draft = f"""# Draft Report — Bagian Track B (Model & Training)

## 1. Metrik (isi bagian \"Akurasi validasi terbaik\")
- CV Macro-F1 (5-fold): **{s['cv_mean_macro_f1']:.4f} ± {s['cv_std_macro_f1']:.4f}**
- OOF overall Macro-F1 (argmax): **{s['oof_overall_macro_f1_argmax']:.4f}**
- Per fold: {', '.join(f"{k}={v:.4f}" for k, v in s['folds'].items())}
(Catatan: metrik lomba Macro-F1 — tulis Macro-F1 dan beri catatan satu baris bahwa metrik mengikuti aturan penilaian.)

## 2. Metodologi (bagian model & training)
- **Backbone:** `{s['backbone']}` — ConvNeXt-Tiny pre-trained ImageNet-12k lalu fine-tuned ImageNet-1k, digunakan via library timm (transfer learning; pre-trained model publik, tidak pernah dilatih pada data kompetisi — sesuai aturan panitia).
- **Validasi:** Stratified Group 5-Fold (group = grup duplikat dari perceptual hashing Track A) — mencegah kebocoran gambar kembar lintas fold.
- **Loss:** Weighted CrossEntropy dengan bobot kelas dari distribusi data (kelas Electronic minoritas mendapat bobot terbesar) + label smoothing {CFG.label_smoothing}.
- **Training:** AdamW (lr {CFG.lr}, wd {CFG.weight_decay}), warmup 1 epoch + cosine decay, {CFG.epochs} epoch, image {CFG.img_size}, batch {CFG.batch}, mixed precision (AMP) + gradient clipping. Seed 42 di semua komponen.
- **Checkpoint:** model terbaik per fold dipilih berdasarkan val Macro-F1.
- **OOF:** probabilitas out-of-fold seluruh 26.527 data latih dikumpulkan sebagai dasar threshold tuning per kelas (Track C).

## 3. Kendala (kontribusi Track B)
- Kuota GPU terbatas (Colab/Kaggle) → mitigasi: AMP, grad checkpointing, backbone Tiny, pembagian fold antar mesin.
- Ketidakseimbangan kelas (Electronic minoritas) → weighted loss + rencana threshold tuning per kelas di OOF.
- (Tambahkan kendala nyata lain yang dialami, mis. disconnect Colab, waktu download dataset.)

## 4. Rencana Pengembangan (kontribusi Track B)
- Ensemble multi-backbone: tambah ConvNeXtV2-Tiny / EfficientNet-B3 untuk diversitas (submission 2–3).
- Naikkan resolusi inference ke 288 (test_input_size bawaan pre-trained ini) — diuji di OOF dulu.
- Eksperimen augmentasi lanjutan (Mixup/CutMix) bila CV menunjukkan overfitting.
- Target: CV Macro-F1 naik ≥ +0.5 poin dari baseline sebelum submission 2.

## 5. Hal untuk Didiskusikan dengan Reviewer
- Apakah threshold tuning per kelas pada OOF berisiko overfit dengan 5-fold?
- Trade-off resolusi (224→288) vs ensemble dengan budget GPU tetap?
- Apakah Stratified Group K-Fold (group = duplikat) sudah cukup untuk data test yang mungkin beda distribusi?
"""
with open('report_track_b_draft.md', 'w') as f:
    f.write(draft)
print(draft)
print('✅ report_track_b_draft.md tersimpan')

In [ ]:
# Fase 3 — DONE CHECK
import os
from config import CFG

checks = {
    '5 checkpoint': all(os.path.exists(f'{CFG.save_dir}/fold{i}.pt') for i in range(5)),
    '5 log fold':   all(os.path.exists(f'{CFG.save_dir}/fold{i}_log.json') for i in range(5)),
    'oof.npy':      os.path.exists(f'{CFG.save_dir}/oof.npy'),
    'oof_meta':     os.path.exists(f'{CFG.save_dir}/oof_meta.json'),
    'cv_summary':   os.path.exists(f'{CFG.save_dir}/cv_summary.json'),
    'environment':  os.path.exists(f'{CFG.save_dir}/environment.json'),
    'REPRODUCE.md': os.path.exists(f'{CFG.save_dir}/REPRODUCE.md'),
}
for name, ok in checks.items():
    print(('✅' if ok else '❌'), name)

if all(checks.values()):
    print('\n🎉 FASE 1–3 SELESAI — GATE 3 siap diumumkan ke Track C')
else:
    print('\n⚠️  Ada yang belum selesai — cek cell yang ❌')